# GHG Emissions - Bronze Layer

## Objective

Read the raw GHG emissions JSON file stored in the Databricks Volume
and create the Bronze layer as a Delta table.

## Data Flow

Raw JSON in Databricks Volume
→ Spark DataFrame
→ Bronze Delta Table

## Source

The data comes from the Paris OpenData dataset:

**Inventaire des émissions de gaz à effet de serre du territoire**

The raw data was previously retrieved from the Paris OpenData API
and stored in the Databricks Volume.

## Input

Raw JSON file:

/Volumes/workspace/default/raw_data/ghg_emissions_raw.json

## Output

Bronze Delta table:

bronze_ghg_emission

## Bronze Layer Principle

The Bronze layer keeps the source data as close as possible to the
original data.

No business transformations or data cleaning are performed at this stage.

The original column order is preserved, and a technical ingestion timestamp
is added to track when the data was loaded into the Bronze layer.

## Processing Steps

1. Read the raw JSON file from the Databricks Volume.
2. Retrieve the original column order from the JSON file.
3. Create a Spark DataFrame.
4. Preserve the original column order.
5. Add _ingestion_timestamp.
6. Write the data as a Delta table named bronze_ghg_emission.
7. Display the Bronze table for verification.

In [0]:
# importing libraries 
from pyspark.sql.functions import current_timestamp
import json

In [0]:
# 1. Path to raw JSON in Volume
RAW_PATH = "/Volumes/workspace/default/raw_data/ghg_emissions_raw.json"

# 2. Read raw JSON
df = spark.read.option("multiline", "true").json(RAW_PATH)

# 3. Keep original column order

with open(RAW_PATH, "r") as f:
    data = json.load(f)

original_columns = list(data[0].keys())

df = df.select(*original_columns)

# 4. Add Bronze metadata
df_bronze = (
    df
    .withColumn("_ingestion_timestamp", current_timestamp())
)







In [0]:
# 5. Write Bronze Delta table
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .saveAsTable("workspace.bronze.bronze_ghg_emission")

In [0]:
# 6. Display Bronze
display(spark.table("bronze.bronze_ghg_emission"))

## Bronze Layer Verification

The Bronze table is displayed below to verify that the raw data has been
successfully loaded from the Volume into the Delta table.

The _ingestion_timestamp column indicates when the records were loaded
into the Bronze layer.